# 💧 AquaSense AI — 06: LIME Local Interpretability Analysis
**Project:** Intelligent Water Quality Assessment and Potability Prediction Using Explainable Machine Learning  

### Overview:
In this notebook, we apply **LIME (Local Interpretable Model-agnostic Explanations)** to fit sparse linear surrogate models around individual water quality predictions.
We evaluate:
- Explanations for 5 correctly predicted potable samples
- Explanations for 5 correctly predicted non-potable samples
- Explanations for 5 misclassified samples (error diagnosis)
- Aggregate global LIME feature importance


In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data.loader import DataLoader
from src.data.preprocessor import WaterQualityPreprocessor
from src.models.trainer import ModelTrainer
from src.explainability.lime_explainer import LIMEExplainer
from src.utils.visualization import set_plot_style

set_plot_style()
print("LIME modules loaded.")


## 1. Load Data & Fit LIME Explainer


In [ ]:
dl = DataLoader(data_path="../data/raw/water_potability.csv")
df = dl.load()
X_train, X_test, y_train, y_test = dl.split(df, test_size=0.20, random_state=42)

preprocessor = WaterQualityPreprocessor.load("../models/preprocessor.pkl")
X_train_proc = preprocessor.transform(X_train)
X_test_proc = preprocessor.transform(X_test)

best_model = ModelTrainer.load_single("best_model", path="../models")

lime_explainer = LIMEExplainer(random_state=42)
lime_explainer.fit(X_train_proc, feature_names=preprocessor.feature_names_out_)
print("LIME explainer initialized.")


## 2. Explain Single Prediction


In [ ]:
sample_idx = 0
sample_row = X_test_proc.iloc[sample_idx]
exp_plot = lime_explainer.plot_explanation(best_model, sample_row, idx=sample_idx)
plt.show()


## 3. Error Analysis: Explaining Misclassified Samples


In [ ]:
y_pred = best_model.predict(X_test_proc)
misclassified_indices = np.where(y_pred != y_test.values)[0]
print(f"Total misclassified test samples: {len(misclassified_indices)}")

for i in misclassified_indices[:3]:
    row = X_test_proc.iloc[i]
    true_cls = y_test.iloc[i]
    pred_cls = y_pred[i]
    print(f"--- Sample #{i} (True: {true_cls}, Pred: {pred_cls}) ---")
    fig = lime_explainer.plot_explanation(best_model, row, idx=i)
    plt.show()


## 4. Aggregate Global LIME Feature Importance


In [ ]:
lime_ranking = lime_explainer.get_feature_ranking(best_model, X_test_proc, n_samples=25)
df_lime = pd.DataFrame(lime_ranking, columns=['Feature', 'Mean Absolute LIME Weight'])
df_lime.head(10)
